In [1]:
import pandas as pd
import numpy as np
from scipy.stats import entropy

In [3]:
df = pd.read_csv("/content/transactions_raw.csv")
df["transaction_timestamp"] = pd.to_datetime(df["transaction_timestamp"])

df = df.sort_values(["user_id", "transaction_timestamp"])
df.head()

,user_id,transaction_date,hour_of_day,transaction_amount,merchant_category,transaction_timestamp,day_of_week,is_weekend,day_of_month,is_night_purchase
0,user_0,2025-01-01,13,90.869306,entertainment,2025-01-01 13:00:00,2,False,1,False
1,user_0,2025-01-05,4,116.611235,travel,2025-01-05 04:00:00,6,True,5,False
2,user_0,2025-01-06,19,47.091855,utilities,2025-01-06 19:00:00,0,False,6,False
3,user_0,2025-01-08,12,168.191071,food_delivery,2025-01-08 12:00:00,2,False,8,False
4,user_0,2025-01-09,0,72.929843,electronics,2025-01-09 00:00:00,3,False,9,True


In [5]:
df["transaction_gap"] = df.groupby("user_id")["transaction_timestamp"] \
    .diff().dt.total_seconds() / 3600  # gap in hours

df["transaction_gap"].fillna(0, inplace=True)

df["transaction_gap_variance"] = df.groupby("user_id")["transaction_gap"] \
    .transform(lambda x: x.rolling(10, min_periods=1).var())

/tmp/ipython-input-393/1590687371.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["transaction_gap"].fillna(0, inplace=True)


In [6]:
df["rolling_7day_spend"] = df.groupby("user_id")["transaction_amount"] \
    .transform(lambda x: x.rolling(7, min_periods=1).mean())

df["rolling_30day_spend"] = df.groupby("user_id")["transaction_amount"] \
    .transform(lambda x: x.rolling(30, min_periods=1).mean())

In [7]:
df["spend_spike_ratio"] = df["transaction_amount"] / \
    (df["rolling_7day_spend"] + 1)

In [8]:
df["burst_score"] = df.groupby("user_id")["transaction_timestamp"] \
    .transform(lambda x: x.diff().dt.total_seconds().lt(3600).astype(int))

df["burst_score"] = df.groupby("user_id")["burst_score"] \
    .transform(lambda x: x.rolling(5, min_periods=1).sum())

In [9]:
df["night_spend_ratio"] = df.groupby("user_id")["is_night_purchase"] \
    .transform(lambda x: x.rolling(30, min_periods=1).mean())

In [12]:
def compute_entropy(x):
    probs = x.value_counts(normalize=True)
    return entropy(probs)

def calculate_user_category_entropy(user_categories):
    entropies = []
    # user_categories is a Series for a single user's merchant_category
    for i in range(len(user_categories)):
        # Define the rolling window based on the original index to align correctly
        # This creates a window of up to 30 previous transactions (including current)
        window = user_categories.iloc[max(0, i - 29):i + 1]
        if len(window) >= 5: # Check min_periods
            entropies.append(compute_entropy(window))
        else:
            entropies.append(np.nan)
    return pd.Series(entropies, index=user_categories.index)

df["category_entropy"] = df.groupby("user_id")["merchant_category"].apply(calculate_user_category_entropy).droplevel(0)

In [13]:
df["is_end_month"] = df["day_of_month"] >= 25

df["monthly_avg_spend"] = df.groupby(["user_id", df["transaction_timestamp"].dt.month]) \
    ["transaction_amount"].transform("mean")

df["end_month_surge_index"] = np.where(
    df["is_end_month"],
    df["transaction_amount"] / (df["monthly_avg_spend"] + 1),
    0
)

In [14]:
df["behavioural_drift_score"] = \
    df["rolling_7day_spend"] - df["rolling_30day_spend"]

In [15]:
df["high_impulse_event"] = np.where(
    (df["spend_spike_ratio"] > 1.8) &
    (df["burst_score"] > 2) &
    (df["category_entropy"] > 1.0),
    1,
    0
)

In [16]:
df.fillna(0, inplace=True)

In [17]:
df.to_csv("transactions_processed.csv", index=False)